# Writing a workflow for the framework: specs and bindings

This notebook is for the authors of workflow packages, such as esssans.
A package that offers a workflow to the framework writes two things, both against `ess.spec`:

- a **spec**: the workflow's name, version, parameters, and outputs. Apps build forms from it, the framework checks requests against it, and records name it. It is all that anyone outside the package sees.
- a **binding**: the code that computes the spec. Apart from the validators of the params model, it is the only code of the package that the backend runs.

`ess.spec` needs only pydantic; `ess.spec.pipeline` adds sciline.
A workflow package depends on `ess.spec`, not on `ess.dispatch`, the client and backend that run it.

The thread: a reduction of one run, its spec and binding, then the same reduction over several runs, summed as they arrive by a *driver*.
At each step, what the framework gives the binding, and what the binding's author promises in return.

## 0. The workflow you have: a sciline pipeline

The toy SANS reduction of `toy_sans.py` stands in for esssans.
A run's data holds its counts per pixel and its monitor count.
`load` stands for loading and correcting a run, the costly step.
The reduction sums the pixels into Q bins (the numerator), divides by the monitor count (the denominator), and subtracts the can's curve from the sample's.

In [1]:
import toy_sans
from toy_sans import (
    CAN_RUNS,
    SAMPLE_RUNS,
    Bins,
    CanRun,
    Denominator,
    IofQ,
    Numerator,
    RunData,
    SampleRun,
    pipeline,
)

print(SAMPLE_RUNS[101], CAN_RUNS[110])
workflow = pipeline.copy()
workflow[RunData[SampleRun]] = SAMPLE_RUNS[101]
workflow[RunData[CanRun]] = CAN_RUNS[110]
workflow[Bins] = 2
workflow.compute(IofQ)

{'counts': [8.0, 6.0, 4.0, 2.0], 'monitor': 10.0} {'counts': [1.0, 1.0, 1.0, 1.0], 'monitor': 10.0}


<scipp.Variable> (Q: 2)    float64  [dimensionless]  [1.2, 0.4]

## 1. The spec of one run

The spec declares the parameters as one pydantic model and the outputs as another.
Each field of the params model is a *parameter*.
A parameter is a *group*, a flat model whose fields are values or data, never another model, or a *table* of such models (section 4).

In [2]:
from pydantic import BaseModel, Field

from ess.spec import Array, ArraySpec, NexusFile, WorkflowSpec

CURVE = ArraySpec(dims=('Q',), unit='dimensionless')


class Run(BaseModel):
    file: NexusFile  # a data field: a raw NeXus file


class Reduction(BaseModel):
    bins: int = Field(default=2, gt=0)


class IofQParams(BaseModel):
    sample: Run
    can: Run
    reduction: Reduction = Field(default_factory=Reduction)


class IofQOutputs(BaseModel):
    iofq: Array(CURVE)  # a data field: a scipp array of this structure


IOFQ = WorkflowSpec(
    name='toy-sans-iofq',
    version=1,
    title='I(Q)',
    description='I(Q) of one sample run, minus that of one can run.',
    params=IofQParams,
    outputs=IofQOutputs,
)

Rules of the spec, each with its reason:

- **A data field holds data, given by reference.** A request names a dataset or another record's output; the binding gets the data read. So the spec says what the data is (`NexusFile`, `Array(...)`), never where it lives.
- **Defaults live in the params model only.** The binding gets every value filled in, so a default in the binding would never be used.
- **A check of one field is a field validator; a check of several is a model validator.** A template checks each value by its own field, before the others are known ([README, Templates](../README.md#templates)).
- **Name and version identify the spec.** Records name them. `version` is the version of the interface: it changes with any change a consumer could observe, in the params model, the outputs, or what the workflow computes. The optional field `code_revision` names the code itself; records do not carry it yet.
- **An `ArraySpec` describes the structure of an array.** Where both an output and the input it feeds declare one, the backend requires them to be equal when a request references the output. Nothing checks the computed value against it.

An app reads the spec, not the package's code: `serialize()` gives a JSON Schema from which a form is built.

In [3]:
schema = IOFQ.serialize().params_schema
{name: field.get('$ref', field) for name, field in schema['properties'].items()}

{'sample': '#/$defs/Run',
 'can': '#/$defs/Run',
 'reduction': '#/$defs/Reduction'}

The params model refuses what cannot run, before anything runs:

In [4]:
from pydantic import ValidationError

from ess.spec import DatasetRef

run = DatasetRef(dataset='run:101')
try:
    IofQParams(sample={'file': run}, can={'file': run}, reduction={'bins': 0})
except ValidationError as error:
    print(error)

1 validation error for IofQParams
reduction.bins
  Input should be greater than 0 [type=greater_than, input_value=0, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/greater_than


## 2. The binding of one run

A binding is what the backend calls to compute a spec's requests.
The simplest binding is a function.
It takes the parameters by name and returns the outputs by name.
It gets each group as a dict of its fields, with every default filled in and every data field read:

In [5]:
from ess.spec import function_binding


def iofq(sample: dict, can: dict, reduction: dict) -> dict:
    workflow = pipeline.copy()
    workflow[RunData[SampleRun]] = sample['file']
    workflow[RunData[CanRun]] = can['file']
    workflow[Bins] = reduction['bins']
    return {'iofq': workflow.compute(IofQ)}


draft = function_binding(IOFQ, iofq)
values = {
    'sample': {'file': SAMPLE_RUNS[101]},
    'can': {'file': CAN_RUNS[110]},
    'reduction': {'bins': 2},
}
draft.compute(values)

{'iofq': <scipp.Variable> (Q: 2)    float64  [dimensionless]  [1.2, 0.4]}

A request with every value given, a *plain request*, goes through `compute(params)`.

For a sciline pipeline, `PipelineBinding` does the same from a table of names: the key that each field of a group sets, and the key that computes each output.

In [6]:
from ess.spec.pipeline import PipelineBinding

single = PipelineBinding(
    IOFQ,
    pipeline,
    params={
        'sample.file': RunData[SampleRun],
        'can.file': RunData[CanRun],
        'reduction.bins': Bins,
    },
    outputs={'iofq': IofQ},
)
single.compute(values)

{'iofq': <scipp.Variable> (Q: 2)    float64  [dimensionless]  [1.2, 0.4]}

Each binding takes its spec, and the backend finds a request's binding by the spec's name and version.

**Promise: a binding modifies none of its inputs, and returns no output that shares memory with an input.**
The framework passes the values it holds without a copy, such as another record's output that a request references.
A binding that modified one, or returned a view of one, would change another record's value.
`forward` in `toy_sans.py` returns `curve['data']['Q', 0].copy()` for this reason.

## 3. Stages: compute the fixed part once

When a user tunes `bins`, the runs stay the same.
The client then keeps a *stage*: a template whose fixed values stay, with *blanks*, the parameters left open (see [client-authors.ipynb](client-authors.ipynb)).
The backend calls `make_stage(fixed, blanks)` once, and the returned callable once per call through the stage, with the values of the blanks.
`PipelineBinding` cuts the pipeline at the keys of the blanks with `sciline.Stage`, and computes what does not depend on them once:

In [7]:
fixed = {key: values[key] for key in ('sample', 'can')}
call = single.make_stage(fixed, ('reduction',))
toy_sans.LOADED.clear()
for bins in (1, 2, 4):
    print(bins, call({'reduction': {'bins': bins}})['iofq'].values)
print('runs loaded:', len(toy_sans.LOADED))

1 [1.6]
2 [1.2 0.4]
4 [0.7 0.5 0.3 0.1]
runs loaded: 2


Two loads for three calls: the sample run and the can run, once each.
A function binding has no `make_stage`, since a stage over a function would hold nothing; the backend refuses a stage over it when the stage is made.

**Promise: a call through a stage gives the outputs of the plain request** (the *caching* symmetry).
The framework records a call through a stage as its plain request, so the record must not depend on how it was computed.
The `sciline.Stage` keeps this promise, given a pure pipeline.
`ess.spec.testing` checks it by computing both, and also checks that no output shares memory with an input:

In [8]:
from ess.spec.testing import check_caching

check_caching(single, fixed, [{'reduction': {'bins': n}} for n in (2, 1, 2)])

## 4. The spec of several runs: tables

Many reductions combine runs: here, several sample runs and several can runs, summed.
A *table* is a list of rows, and a row is a flat model too.
The row is what the reduction repeats per run, so the same `Run` is a group of the single-run spec and a row of the multi-run spec:

In [9]:
class MultiIofQParams(BaseModel):
    sample: list[Run] = Field(min_length=1)
    can: list[Run] = Field(min_length=1)
    reduction: Reduction = Field(default_factory=Reduction)


IOFQ_MULTI = WorkflowSpec(
    name='toy-sans-iofq-multi',
    version=1,
    title='I(Q) of summed runs',
    description='I(Q) of the summed sample runs, minus that of the summed can runs.',
    params=MultiIofQParams,
    outputs=IofQOutputs,
)

Rules of tables:

- **A row holds what belongs to one run**, such as a sample run and its own transmission run. Rows of different tables are independent; pairing runs into rows is the app's job.
- **Structure below a row is the binding's**, such as detector banks. The framework sees rows only.
- **The multi-run spec declares the same outputs model as the single-run spec**, and with one row per table gives the same outputs (the *one-row* symmetry).
- **We recommend that the multi-run spec name each table as the single-run spec names the group**, with the same row type: `sample: list[Run]` for `sample: Run`. A reader then sees at once which table holds the runs of which group.

## 5. A driver: summing runs as they arrive

When runs arrive one at a time, a user wants the sum so far after each, without reducing the earlier runs again.
The client's *accumulator* takes one *push* at a time, each one or more rows of one table (see [client-authors.ipynb](client-authors.ipynb)).
Behind it is a *driver*: an object with `push(table, rows)` and `outputs()`, which the workflow package writes for one spec.
Only a driver accumulates: the framework never decides how rows combine.

This driver sums each run's numerator and denominator per table, and computes I(Q) once from the sums.
It cuts the one-run pipeline with `sciline.Stage` twice: at a run's data, to compute one run's numerator and denominator, and at the four sums, to compute I(Q).
It gets the fixed values that are not tables, here `reduction`, when it is made:

In [10]:
import functools
from collections.abc import Mapping, Sequence
from typing import Any

import sciline


class IofQDriver:
    def __init__(self, pipeline: sciline.Pipeline, fixed: Mapping[str, Any]) -> None:
        workflow = pipeline.copy()
        workflow[Bins] = fixed['reduction']['bins']
        self._contribute = {
            table: (
                RunData[run],
                sciline.Stage(
                    workflow,
                    inputs=(RunData[run],),
                    outputs=(Numerator[run], Denominator[run]),
                ),
            )
            for table, run in (('sample', SampleRun), ('can', CanRun))
        }
        sums = [key[run] for key in (Numerator, Denominator) for run in (SampleRun, CanRun)]
        self._finalize = sciline.Stage(workflow, inputs=sums, outputs=(IofQ,))
        self._sums: dict[str, dict] = {}  # per table, by key

    def push(self, table: str, rows: Sequence[Mapping[str, Any]]) -> None:
        key, stage = self._contribute[table]
        for row in rows:
            parts = stage.compute({key: row['file']})
            old = self._sums.get(table)
            self._sums[table] = (
                parts if old is None else {k: old[k] + v for k, v in parts.items()}
            )

    def outputs(self) -> dict[str, Any]:
        if empty := sorted(self._contribute.keys() - self._sums.keys()):
            raise ValueError(f'no rows in {empty} yet')
        sums = {k: v for parts in self._sums.values() for k, v in parts.items()}
        return {'iofq': self._finalize.compute(sums)[IofQ]}


make_driver = functools.partial(IofQDriver, pipeline)
driver = make_driver({'reduction': {'bins': 2}})
toy_sans.LOADED.clear()
driver.push('sample', [{'file': SAMPLE_RUNS[101]}])
driver.push('can', [{'file': CAN_RUNS[110]}])
print(driver.outputs()['iofq'].values)
driver.push('sample', [{'file': SAMPLE_RUNS[102]}, {'file': SAMPLE_RUNS[103]}])
print(driver.outputs()['iofq'].values)
print('runs loaded:', len(toy_sans.LOADED))

[1.2 0.4]
[0.85 0.25]
runs loaded: 4


Four loads for four runs: a push loads only its own rows.
Summing numerators and denominators apart, and dividing once, is what makes a sum of runs correct; a sum of normalized curves would not be.
The driver needs no spec, backend, or client, so a package uses and tests it as plain code.

The author of a driver promises, and the backend does not check:

- **`outputs()` gives the outputs of every row pushed so far**, however the rows were grouped into pushes, and whatever the order of pushes into different tables (the *arrival* and *order* symmetries). So each sum may depend on the fixed values and on one table only.
- **Within a table, the rows keep the order in which they were pushed.**
- **`push` does not modify a row.** A row may be another record's output. A driver that adds in place copies the first contribution, so that adding never changes a value that another record returned. `IofQDriver` adds with `a + b`, which modifies nothing.
- **A driver may refuse a state**, such as one with no can row yet, by raising in `outputs()`. Later pushes may make the state valid.
- **Validators that read several values, such as the params model's own, change no value.** The driver gets each fixed value checked by its own field alone.

In return, `outputs()` may share memory with what the driver holds: the backend is done with the outputs of a state before it pushes the next rows.

`check_arrival_and_order` takes the function that makes a driver, the fixed values that are not tables, and the rows of each table.
It pushes the rows in several schedules, one row at a time and in groups, table by table and alternating, and reads between pushes.
It compares each state with what a new driver gives after one push per table, skipping states that the driver refuses.
It also fails if a push modifies its rows:

In [11]:
from ess.spec.testing import check_arrival_and_order

tables = {
    'sample': [{'file': SAMPLE_RUNS[run]} for run in (101, 102, 103)],
    'can': [{'file': CAN_RUNS[run]} for run in (110, 111)],
}
check_arrival_and_order(make_driver, {'reduction': {'bins': 2}}, tables)

Both sides of the comparison go through the driver, so the check does not prove the outputs right.

A sum that depends on two tables would break the order symmetry: the can's sum would change with the sample rows pushed before it.
esssans has this case, since its pixel masks come from the sample run.
Its driver must give outputs that do not depend on the order of pushes, for example by keeping the can rows that a mask applies to; `check_arrival_and_order` catches a driver whose outputs do.

A workflow that needs every row at once, such as a fit over all runs, writes a driver that keeps its rows and computes over all of them in `outputs()`.
The outputs are the same as for a driver that sums; only the cost differs, and which to pay is the workflow's choice.

## 6. The binding of several runs

`DriverBinding` binds the multi-run spec to the function that makes a driver.
A plain request makes a driver, pushes each table's rows in one push, and reads the outputs:

In [12]:
from ess.spec import DriverBinding

multi = DriverBinding(IOFQ_MULTI, make_driver)
multi.compute({**tables, 'reduction': {'bins': 2}})

{'iofq': <scipp.Variable> (Q: 2)    float64  [dimensionless]  [0.85, 0.25]}

For an accumulator, the backend calls `make_accumulator(fixed, blanks)`.
The binding makes a driver from the fixed values that are not tables, and pushes the rows of each table in `fixed`, such as a can run fixed by the client's template.
The backend then pushes each client push into the driver:

In [13]:
driver = multi.make_accumulator(
    {'reduction': {'bins': 2}, 'can': [{'file': CAN_RUNS[110]}]}, ('sample',)
)
driver.push('sample', [{'file': SAMPLE_RUNS[101]}])
driver.outputs()

{'iofq': <scipp.Variable> (Q: 2)    float64  [dimensionless]  [1.2, 0.4]}

**Promise: one row per table gives the outputs of the single-run spec** (the *one-row* symmetry).
The package keeps it: the driver must compute what the pipeline computes.
`check_one_row` is the only check against an independent reference:

In [14]:
from ess.spec.testing import check_one_row

check_one_row(
    single,
    multi,
    values,
    {'sample': [values['sample']], 'can': [values['can']], 'reduction': {'bins': 2}},
)

With the naming convention of section 4, the one-row values are the single-run values with each group in a list.

Each binding serves some of the client's actions, and the backend refuses the others when a stage or an accumulator is made:

| Binding | Plain request | Stage | Accumulator |
|---|---|---|---|
| `PipelineBinding` | `compute` | `make_stage` | refused |
| `DriverBinding` | `compute` | refused | `make_accumulator` |
| `function_binding` | `compute` | refused | refused |

## 7. What the framework gives the binding

The binding may rely on this:

| Call | When, and with what |
|---|---|
| `compute(params)` | once per plain request; every value, with data read and defaults filled in; possibly from several threads at once |
| `make_stage(fixed, blanks)` | once per stage of a client, at its first call; every value but the blanks; possibly from several threads at once |
| the callable it returns | once per call through the stage, with the values of the blanks; possibly from several threads at once |
| `make_accumulator(fixed, blanks)` | once per accumulator; every value but the tables that the pushes fill. `DriverBinding` calls its function that makes a driver here, and in `compute`, possibly from several threads at once |
| `driver.push(table, rows)` | once per client push, with its rows, in the order logged; one thread at a time; never while the readers of the state before it are busy |
| `driver.outputs()` | once per state that someone reads; one thread at a time |

[framework-promises.ipynb](framework-promises.ipynb) traces these calls for one accumulator.

## Summary: what the author of a workflow promises

| Promise | Checked by |
|---|---|
| defaults live in the params model only | nobody |
| a check of several values is a model validator, and changes no value | nobody |
| a new version when the params or outputs models change | nobody |
| the binding returns no output that shares memory with an input | `check_one_row`, `check_caching` (numpy-backed values) |
| the binding modifies no input | nobody, except for the rows a push gets (`check_arrival_and_order`) |
| the pipeline is pure, so a call through a stage gives the outputs of the plain request | `check_caching` |
| the multi-run spec with one row per table gives the single-run spec's outputs | `check_one_row` |
| after each push, the driver's outputs are those of all rows pushed so far, in any grouping and in any order across tables | `check_arrival_and_order` |
| `push` modifies no row | `check_arrival_and_order` |
| structure below a row belongs to the binding | nobody |

A package registers its bindings through the entry-point group `ess.dispatch.workflows` (decided, not implemented).
Until then, a notebook or app passes them to `local(proposal=..., datasets=..., bind=[single, multi, forward_binding])`.